# 有馬記念用の機械学習モデル（2025年〜2023年あたりのデータを活用）してG１〜３のレースをメインにやってみる

In [1]:
import pandas as pd
import numpy as np
import datetime
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
#import lightgbm as lgb
import requests
from bs4 import BeautifulSoup
import time
import re
from urllib.request import Request, urlopen
#import optuna.integration.lightgbm as lgb_o
from itertools import combinations, permutations
import matplotlib.pyplot as plt


In [2]:
import random

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:115.0) Gecko/20100101 Firefox/115.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:115.0) Gecko/20100101 Firefox/115.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.1.2 Safari/605.1.15",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Edg/115.0.0.0",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 OPR/85.0.4341.72",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 OPR/85.0.4341.72",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Vivaldi/5.3.2679.55",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Vivaldi/5.3.2679.55",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Brave/1.40.107",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Brave/1.40.107",
]

random.choice(USER_AGENTS)

'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Edg/115.0.0.0'

In [3]:
class Results:
    @staticmethod
    def scrape(race_id_list):
        """
        レース結果データをスクレイピングする関数
        Parameters:
        ----------
        race_id_list : list
            レースIDのリスト
        Returns:
        ----------
        race_results_df : pandas.DataFrame
            全レース結果データをまとめてDataFrame型にしたもの
        """
        #race_idをkeyにしてDataFrame型を格納
        race_results = {}
        for race_id in tqdm(race_id_list):
            time.sleep(1)
            try:
                url = "https://db.netkeiba.com/race/" + race_id
                headers = {'User-Agent': random.choice(USER_AGENTS)}
                html = requests.get(url, headers=headers)
                html.encoding = "EUC-JP"
                # メインとなるテーブルデータを取得
                df = pd.read_html(html.text)[0]
                # 列名に半角スペースがあれば除去する
                df = df.rename(columns=lambda x: x.replace(' ', ''))
                # 天候、レースの種類、コースの長さ、馬場の状態、日付をスクレイピング
                soup = BeautifulSoup(html.text, "html.parser")
                texts = (
                    soup.find("div", attrs={"class": "data_intro"}).find_all("p")[0].text
                    + soup.find("div", attrs={"class": "data_intro"}).find_all("p")[1].text
                )
                info = re.findall(r'\w+', texts)
                for text in info:
                    if text in ["芝", "ダート"]:
                        df["race_type"] = [text] * len(df)
                    if "障" in text:
                        df["race_type"] = ["障害"] * len(df)
                    if "m" in text:
                        df["course_len"] = [int(re.findall(r"\d+", text)[-1])] * len(df)
                    if text in ["良", "稍重", "重", "不良"]:
                        df["ground_state"] = [text] * len(df)
                    if text in ["曇", "晴", "雨", "小雨", "小雪", "雪"]:
                        df["weather"] = [text] * len(df)
                    if "年" in text:
                        df["date"] = [text] * len(df)
                #馬ID、騎手IDをスクレイピング
                horse_id_list = []
                horse_a_list = soup.find("table", attrs={"summary": "レース結果"}).find_all(
                    "a", attrs={"href": re.compile("^/horse")}
                )
                for a in horse_a_list:
                    horse_id = re.findall(r"\d+", a["href"])
                    horse_id_list.append(horse_id[0])
                jockey_id_list = []
                jockey_a_list = soup.find("table", attrs={"summary": "レース結果"}).find_all(
                    "a", attrs={"href": re.compile("^/jockey")}
                )
                for a in jockey_a_list:
                    jockey_id = re.findall(r"\d+", a["href"])
                    jockey_id_list.append(jockey_id[0])
                df["horse_id"] = horse_id_list
                df["jockey_id"] = jockey_id_list
                #インデックスをrace_idにする
                df.index = [race_id] * len(df)
                race_results[race_id] = df
            #存在しないrace_idを飛ばす
            except IndexError:
                continue
            except AttributeError: #存在しないrace_idでAttributeErrorになるページもあるので追加
                continue
            #wifiの接続が切れた時などでも途中までのデータを返せるようにする
            except Exception as e:
                print(e)
                break
            #Jupyterで停止ボタンを押した時の対処
            except:
                break
        #pd.DataFrame型にして一つのデータにまとめる
        race_results_df = pd.concat([race_results[key] for key in race_results])
        return race_results_df

In [4]:
import pandas as pd
import requests
import random

# ユーザーエージェントのリスト（適宜追加してください）
USER_AGENTS = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
]

# 取得したいURLのリスト
urls = [
    # 1ページ目 (page指定なし)
    'https://db.netkeiba.com/?pid=race_list&word=&track%5B%5D=1&start_year=2024&start_mon=1&end_year=2024&end_mon=12&jyo%5B%5D=01&jyo%5B%5D=02&jyo%5B%5D=03&jyo%5B%5D=04&jyo%5B%5D=05&jyo%5B%5D=06&jyo%5B%5D=07&jyo%5B%5D=08&jyo%5B%5D=09&jyo%5B%5D=10&jyo%5B%5D=30&jyo%5B%5D=35&jyo%5B%5D=36&jyo%5B%5D=42&jyo%5B%5D=43&jyo%5B%5D=44&jyo%5B%5D=45&jyo%5B%5D=46&jyo%5B%5D=47&jyo%5B%5D=48&jyo%5B%5D=50&jyo%5B%5D=51&jyo%5B%5D=54&jyo%5B%5D=55&jyo%5B%5D=65&grade%5B%5D=1&grade%5B%5D=2&grade%5B%5D=3&kyori_min=&kyori_max=&sort=date&list=100',
    # 2ページ目 (page=2)
    "https://db.netkeiba.com//?pid=race_list&word=&track%5B0%5D=1&start_year=2024&start_mon=1&end_year=2024&end_mon=12&jyo%5B0%5D=01&jyo%5B1%5D=02&jyo%5B2%5D=03&jyo%5B3%5D=04&jyo%5B4%5D=05&jyo%5B5%5D=06&jyo%5B6%5D=07&jyo%5B7%5D=08&jyo%5B8%5D=09&jyo%5B9%5D=10&jyo%5B10%5D=30&jyo%5B11%5D=35&jyo%5B12%5D=36&jyo%5B13%5D=42&jyo%5B14%5D=43&jyo%5B15%5D=44&jyo%5B16%5D=45&jyo%5B17%5D=46&jyo%5B18%5D=47&jyo%5B19%5D=48&jyo%5B20%5D=50&jyo%5B21%5D=51&jyo%5B22%5D=54&jyo%5B23%5D=55&jyo%5B24%5D=65&grade%5B0%5D=1&grade%5B1%5D=2&grade%5B2%5D=3&kyori_min=&kyori_max=&sort=date&list=100&page=2"
]

# データフレームを格納するリスト
df_list = []

for url in urls:
    try:
        headers = {'User-Agent': random.choice(USER_AGENTS)}
        
        # データの取得
        response = requests.get(url, headers=headers)
        response.encoding = "EUC-JP" # 文字化け対策
        
        # テーブルデータの読み込み
        # pd.read_htmlはリストを返すため、[0]で最初のテーブルを取得
        dfs = pd.read_html(response.text)
        
        if len(dfs) > 0:
            df_temp = dfs[0]
            df_list.append(df_temp)
            print(f"取得成功: {url[:60]}... (行数: {len(df_temp)})")
        else:
            print(f"テーブルが見つかりませんでした: {url}")
            
    except Exception as e:
        print(f"エラーが発生しました: {e} URL: {url}")

# 結合処理
if df_list:
    # 複数のDataFrameを縦に結合 (ignore_index=Trueでindexを振り直す)
    final_df = pd.concat(df_list, ignore_index=True)
    
    # 列名の半角スペース除去
    final_df = final_df.rename(columns=lambda x: x.replace(' ', ''))
    
    print("-" * 30)
    print(f"統合完了。合計行数: {len(final_df)}")
    print(final_df.head()) # 確認用表示
else:
    print("データが取得できませんでした。")

/var/folders/6g/w4z9zgn15ng21w8fwdt1bkpr0000gn/T/ipykernel_42383/2619836402.py:32: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(response.text)


取得成功: https://db.netkeiba.com/?pid=race_list&word=&track%5B%5D=1&s... (行数: 100)
取得成功: https://db.netkeiba.com//?pid=race_list&word=&track%5B0%5D=1... (行数: 14)
------------------------------
統合完了。合計行数: 114
          開催日    開催 天気   R            レース名  映像     距離  頭数 馬場     タイム  \
0  2024/12/28  5中山9  晴  11      ホープフルS(GI) NaN  芝2000  18  良  2:00.5   
1  2024/12/22  5中山8  晴  11        有馬記念(GI) NaN  芝2500  15  良  2:31.8   
2  2024/12/21  7京都7  曇  11        阪神C(GII) NaN  芝1400  18  良  1:20.1   
3  2024/12/15  7京都6  晴  11  朝日フューチュリティ(GI) NaN  芝1600  16  良  1:34.1   
4  2024/12/14  5中山5  晴  11    ターコイズS(GIII) NaN  芝1600  16  良  1:33.2   

         ペース       勝ち馬    騎手      調教師        2着馬        3着馬  
0  36.0-35.5  クロワデュノール  北村友一  [西]斉藤崇史      ジョバンニ  ファウストラーゼン  
1  31.4-35.2     レガレイラ  戸崎圭太  [東]木村哲也    シャフリヤール    ダノンデサイル  
2  34.5-34.4    ナムラクレア  ルメール  [西]長谷川浩     マッドクール     オフトレイル  
3  35.4-33.7  アドマイヤズーム  川田将雅  [西]友道康夫  ミュージアムマイル   ランスオブカオス  
4  34.7-35.1     アルジーヌ  西村淳也  [西]中内田充  ビヨンドザヴァレー    

/var/folders/6g/w4z9zgn15ng21w8fwdt1bkpr0000gn/T/ipykernel_42383/2619836402.py:32: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(response.text)


In [6]:
final_df

,開催日,開催,天気,R,レース名,映像,距離,頭数,馬場,タイム,ペース,勝ち馬,騎手,調教師,2着馬,3着馬
0,2024/12/28,5中山9,晴,11,ホープフルS(GI),NaN,芝2000,18,良,2:00.5,36.0-35.5,クロワデュノール,北村友一,[西]斉藤崇史,ジョバンニ,ファウストラーゼン
1,2024/12/22,5中山8,晴,11,有馬記念(GI),NaN,芝2500,15,良,2:31.8,31.4-35.2,レガレイラ,戸崎圭太,[東]木村哲也,シャフリヤール,ダノンデサイル
2,2024/12/21,7京都7,曇,11,阪神C(GII),NaN,芝1400,18,良,1:20.1,34.5-34.4,ナムラクレア,ルメール,[西]長谷川浩,マッドクール,オフトレイル
3,2024/12/15,7京都6,晴,11,朝日フューチュリティ(GI),NaN,芝1600,16,良,1:34.1,35.4-33.7,アドマイヤズーム,川田将雅,[西]友道康夫,ミュージアムマイル,ランスオブカオス
4,2024/12/14,5中山5,晴,11,ターコイズS(GIII),NaN,芝1600,16,良,1:33.2,34.7-35.1,アルジーヌ,西村淳也,[西]中内田充,ビヨンドザヴァレー,ドゥアイズ
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,2024/01/13,1小倉1,晴,11,愛知杯(GIII),NaN,芝2000,14,良,1:57.9,33.5-35.8,ミッキーゴージャス,川田将雅,[西]安田隆行,タガノパッション,コスタボニータ
110,2024/01/08,1京都3,晴,11,日刊スポシンザン記念(GIII),NaN,芝1600,18,良,1:34.5,34.3-36.1,ノーブルロジャー,川田将雅,[西]吉岡辰弥,エコロブルーム,ウォーターリヒト
111,2024/01/07,1中山2,曇,11,フェアリーS(GIII),NaN,芝1600,14,良,1:34.0,34.7-35.1,イフェイオン,西村淳也,[西]杉山佳明,マスクオールウィン,ラヴスコール
112,2024/01/06,1中山1,晴,11,日刊スポ賞中山金杯(GIII),NaN,芝2000,17,良,1:58.9,36.1-34.8,リカンカブール,津村明秀,[西]田中克典,ククナ,マイネルクリソーラ


In [17]:
import pandas as pd
import re

# 1. 場所名とネット競馬ID(2桁)の対応辞書
# ユーザー様のリスト順序等を考慮し、ネット競馬で実際に使われるコードを定義しています
place_map = {
    # JRA (01-10)
    '札幌': '01', '函館': '02', '福島': '03', '新潟': '04', '東京': '05', 
    '中山': '06', '中京': '07', '京都': '08', '阪神': '09', '小倉': '10',
    # NAR・地方 (30-65) ※主な場所のID
    '門別': '30', '盛岡': '35', '水沢': '36', 
    '浦和': '42', '船橋': '43', '大井': '44', '川崎': '45', 
    '金沢': '46', '笠松': '47', '名古屋': '48', 
    '園田': '50', '姫路': '51', 
    '高知': '54', '佐賀': '55', 
    '帯広(ば)': '65', '帯広': '65' # 表記揺れ対応
}

def generate_race_id(row):
    """
    行ごとのデータからRaceID(2025xxxxxxxx)を生成する関数
    想定入力: 開催='5阪神4' または '5回阪神4日', R=11
    """
    try:
        # 開催情報の文字列取得
        kaisai_str = str(row['開催'])
        
        # 正規表現で「数字」「場所名」「数字」を抽出
        # 例: "5阪神4" -> group1=5, group2=阪神, group3=4
        # "回"や"日"が含まれていても無視して数字と場所だけ抜くように調整
        match = re.search(r'(\d+)[回]*(\D+?)(\d+)[日]*', kaisai_str)
        
        if match:
            kai = match.group(1).zfill(2)   # 回 (5 -> 05)
            place_name = match.group(2)     # 場所名 (阪神)
            day = match.group(3).zfill(2)   # 日 (4 -> 04)
            
            # 場所名からIDを取得
            place_id = place_map.get(place_name)
            
            # レース番号 (11 -> 11)
            race_num = str(row['R']).zfill(2)
            
            if place_id:
                # 2025 + 場所ID + 回 + 日 + レース番号
                # 例: 2025 + 09 + 05 + 04 + 11
                return "2021" + place_id + kai + day + race_num
            else:
                return None # 対応する場所名がない場合
        else:
            return None # マッチしない場合
            
    except Exception as e:
        return None

# 2. 関数を適用して新しい列 'race_id' を作成
final_df['race_id'] = final_df.apply(generate_race_id, axis=1)

# 確認用: 変換できたデータと元の列を表示
print(final_df[['開催', 'R', 'race_id']].head(10))

# race_idが作成できたものだけを抽出したい場合
# final_df = final_df.dropna(subset=['race_id'])

     開催   R       race_id
0  5中山9  11  202106050911
1  5中山8  11  202106050811
2  7京都7  11  202108070711
3  7京都6  11  202108070611
4  5中山5  11  202106050511
5  7京都4  11  202108070411
6  4中京3  11  202107040311
7  5中山1  11  202106050111
8  7京都1  11  202108070111
9  5東京8  12  202105050812


In [18]:
race_id_list = final_df['race_id']
results = Results.scrape(race_id_list)

  0%|          | 0/114 [00:00<?, ?it/s]

/var/folders/6g/w4z9zgn15ng21w8fwdt1bkpr0000gn/T/ipykernel_42383/989785255.py:25: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(html.text)[0]
/var/folders/6g/w4z9zgn15ng21w8fwdt1bkpr0000gn/T/ipykernel_42383/989785255.py:25: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(html.text)[0]
/var/folders/6g/w4z9zgn15ng21w8fwdt1bkpr0000gn/T/ipykernel_42383/989785255.py:25: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(html.text)[0]
/var/folders/6g/w4z9zgn15ng21w8fwdt1bkpr0000gn/T/ipykernel_42383/989785255.py:25: FutureWarning: Passing literal html to 'read_html' is deprecate

In [19]:
results.to_pickle('2021_results.pickle')
#results = pd.read_pickle('results.pickle')

In [30]:
import pandas as pd
import re

# 1. 場所名とネット競馬ID(2桁)の対応辞書
# ユーザー様のリスト順序等を考慮し、ネット競馬で実際に使われるコードを定義しています
place_map = {
    # JRA (01-10)
    '札幌': '01', '函館': '02', '福島': '03', '新潟': '04', '東京': '05', 
    '中山': '06', '中京': '07', '京都': '08', '阪神': '09', '小倉': '10',
    # NAR・地方 (30-65) ※主な場所のID
    '門別': '30', '盛岡': '35', '水沢': '36', 
    '浦和': '42', '船橋': '43', '大井': '44', '川崎': '45', 
    '金沢': '46', '笠松': '47', '名古屋': '48', 
    '園田': '50', '姫路': '51', 
    '高知': '54', '佐賀': '55', 
    '帯広(ば)': '65', '帯広': '65' # 表記揺れ対応
}

def generate_race_id(row):
    """
    行ごとのデータからRaceID(2025xxxxxxxx)を生成する関数
    想定入力: 開催='5阪神4' または '5回阪神4日', R=11
    """
    try:
        # 開催情報の文字列取得
        kaisai_str = str(row['開催'])
        
        # 正規表現で「数字」「場所名」「数字」を抽出
        # 例: "5阪神4" -> group1=5, group2=阪神, group3=4
        # "回"や"日"が含まれていても無視して数字と場所だけ抜くように調整
        match = re.search(r'(\d+)[回]*(\D+?)(\d+)[日]*', kaisai_str)
        
        if match:
            kai = match.group(1).zfill(2)   # 回 (5 -> 05)
            place_name = match.group(2)     # 場所名 (阪神)
            day = match.group(3).zfill(2)   # 日 (4 -> 04)
            
            # 場所名からIDを取得
            place_id = place_map.get(place_name)
            
            # レース番号 (11 -> 11)
            race_num = str(row['R']).zfill(2)
            
            if place_id:
                # 2025 + 場所ID + 回 + 日 + レース番号
                # 例: 2025 + 09 + 05 + 04 + 11
                return "2017" + place_id + kai + day + race_num
            else:
                return None # 対応する場所名がない場合
        else:
            return None # マッチしない場合
            
    except Exception as e:
        return None

# 2. 関数を適用して新しい列 'race_id' を作成
final_df['race_id'] = final_df.apply(generate_race_id, axis=1)

# 確認用: 変換できたデータと元の列を表示
print(final_df[['開催', 'R', 'race_id']].head(10))

# race_idが作成できたものだけを抽出したい場合
# final_df = final_df.dropna(subset=['race_id'])

     開催   R       race_id
0  5中山9  11  201706050911
1  5中山8  11  201706050811
2  7京都7  11  201708070711
3  7京都6  11  201708070611
4  5中山5  11  201706050511
5  7京都4  11  201708070411
6  4中京3  11  201707040311
7  5中山1  11  201706050111
8  7京都1  11  201708070111
9  5東京8  12  201705050812


In [31]:
race_id_list = final_df['race_id']
results = Results.scrape(race_id_list)

  0%|          | 0/114 [00:00<?, ?it/s]

/var/folders/6g/w4z9zgn15ng21w8fwdt1bkpr0000gn/T/ipykernel_42383/989785255.py:25: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(html.text)[0]
/var/folders/6g/w4z9zgn15ng21w8fwdt1bkpr0000gn/T/ipykernel_42383/989785255.py:25: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(html.text)[0]
/var/folders/6g/w4z9zgn15ng21w8fwdt1bkpr0000gn/T/ipykernel_42383/989785255.py:25: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(html.text)[0]
/var/folders/6g/w4z9zgn15ng21w8fwdt1bkpr0000gn/T/ipykernel_42383/989785255.py:25: FutureWarning: Passing literal html to 'read_html' is deprecate

In [32]:
results.to_pickle('2017_results.pickle')
#results = pd.read_pickle('results.pickle')